# Анализ правил разметки и сложных случаев (Labeling Guide & Hard Cases)
**Проект:** VolgaFinal (Волга-IT 2026. Финал. «ИИ и анализ данных»)  
**Основа:** `labeling_guide.md` и спецификация `.context/specs/labeling.md`  
**Цель исследования:** Детальный разбор инструкций разметки на реальных данных `dev_labels.csv` и `train.csv`. Анализ 7 категорий сложных случаев (Критерий 3 регламента — 15 баллов): `sarcasm`, `obscene`, `typos`, `jargon`, `emoji`, `mixed`, `context`.

### Структура ноутбука:
1. Загрузка данных через единый `DataLoader`
2. Валидация инвариантов разметки (`is_service` и вложенность `aggression`)
3. Разбор сложных тегов на реальных примерах:
   - `sarcasm` (сарказм, ирония, насмешки)
   - `obscene` (обсценная лексика, направленный мат vs экспрессия)
   - `jargon` (жаргон и аббревиатуры ЖКХ Ленинградской области)
   - `typos` (опечатки, отсутствие пунктуации)
   - `emoji` (эмодзи-реакции и смена тональности)
   - `mixed` (смешанная тональность, правило союза «но»)
   - `context` (контекстно-зависимые реплики)
4. Формирование правил и эвристик для будущего препроцессора и классификаторов

In [1]:
import sys
import re
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 140)

sys.path.insert(0, '..')
from src.data.loader import load_train, load_dev

# Загрузка через единый DataLoader с нормализацией времени
train = load_train('../data')
dev = load_dev('../data')

print(f'Train загружен: {train.shape}, Dev загружен: {dev.shape}')

## 1. Проверка жестких инвариантов разметки
Согласно `labeling_guide.md`:
1. **Служебные сообщения (`is_service = 1`):** строго `sentiment = neutral`, `aggression = 0`.
2. **Агрессия (`aggression = 1`):** всегда сочетается только с `sentiment = negative`.

In [2]:
# Инвариант 1: Служебные сообщения
train_svc_viol = train[(train['is_service'] == 1) & ((train['sentiment'] != 'neutral') | (train['aggression'] != 0))]
dev_svc_viol = dev[(dev['is_service'] == 1) & ((dev['sentiment'] != 'neutral') | (dev['aggression'] != 0))]
print(f'Нарушений инварианта служебных сообщений в train: {len(train_svc_viol)}')
print(f'Нарушений инварианта служебных сообщений в dev: {len(dev_svc_viol)}')

# Инвариант 2: Вложенность агрессии
train_agg_viol = train[(train['sentiment'] != 'negative') & (train['aggression'] == 1)]
dev_agg_viol = dev[(dev['sentiment'] != 'negative') & (dev['aggression'] == 1)]
print(f'Агрессия при sentiment != negative в train: {len(train_agg_viol)}')
print(f'Агрессия при sentiment != negative в dev: {len(dev_agg_viol)}')

print('\nВывод: Инварианты соблюдаются абсолютно строго (100% соответствие).')

## 2. Разбор сложных тегов: `sarcasm` (47 сообщений в dev)
**Определение:** Ирония или сарказм, где буквальный смысл противоположен реальному эмоциональному посылу.  
**Паттерны:** Формально вежливые слова («спасибо», «молодцы», «all inclusive») в сочетании с описанием неработающих служб, возмущением или смеющимися/гневными смайлами (`🤣`, `😡`).

In [3]:
sarcasm_df = dev[dev['tags'].fillna('').str.contains('sarcasm')]
print(f'Всего саркастических сообщений в dev: {len(sarcasm_df)}')
print('Распределение тональности в sarcasm:')
print(sarcasm_df['sentiment'].value_counts())

print('\nПримеры сарказма из реального dev-сета:')
for idx, r in sarcasm_df[['text', 'sentiment', 'aggression']].head(5).iterrows():
    print(f'- [{r["sentiment"]}, agg={r["aggression"]}] {r["text"]}')

## 3. Разбор сложных тегов: `obscene` (Обсценная лексика и мат)
**Критическое разграничение:**
- **Мат как агрессия (`aggression = 1`):** Направлен на человека (соседа, диспетчера, УК) с целью унизить («руко@опые», «вы тупая»).
- **Мат как экспрессия (`aggression = 0`):** Междометие, выражение досады по поводу обстоятельств без конкретного адресата («Пипец!», «срём где живём»).

In [4]:
obscene_df = dev[dev['tags'].fillna('').str.contains('obscene')]
print(f'Всего сообщений с обсценной лексикой в dev: {len(obscene_df)}')
print('Распределение агрессии среди obscene:')
print(obscene_df['aggression'].value_counts())

print('\nПримеры обсценной лексики с aggression=1 (направленный выпад):')
for idx, r in obscene_df[obscene_df['aggression'] == 1].head(3).iterrows():
    print(f'- {r["text"]}')

print('\nПримеры обсценной лексики с aggression=0 (ненаправленная экспрессия):')
for idx, r in obscene_df[obscene_df['aggression'] == 0].head(3).iterrows():
    print(f'- {r["text"]}')

## 4. Разбор сложных тегов: `jargon` (ЖКХ-сленг и терминология)
Сообщения с доменной лексикой ЖКХ (`ГВС`, `ХВС`, `парадная`, `ОДН`, `ИПУ`, `РСО`, `УК`).  
Особое внимание: регионализм **«парадная»** — стандартный термин жителей СПб и Ленинградской области для подъезда.

In [5]:
jargon_df = dev[dev['tags'].fillna('').str.contains('jargon')]
print(f'Всего сообщений с ЖКХ-жаргоном в dev: {len(jargon_df)}')
print('Распределение тональности в jargon:')
print(jargon_df['sentiment'].value_counts())

# Поиск ключевых терминов ЖКХ в train и dev
terms = ['парадная', 'гвс', 'хвс', 'ук', 'одн', 'ипу', 'лифт', 'счетчик']
term_counts = {}
for t in terms:
    c_train = train['text'].str.lower().str.contains(t).sum()
    c_dev = dev['text'].str.lower().str.contains(t).sum()
    term_counts[t] = {'Train': c_train, 'Dev': c_dev}

print('\nВстречаемость ключевых ЖКХ-терминов:')
print(pd.DataFrame(term_counts).T)

## 5. Разбор сложных тегов: `mixed` (Смешанная полярность)
**Правило разметки:** Размечается по **итоговому смыслу** сообщения.  
Если в реплике сначала идет формальная благодарность («Спасибо, что починили»), но затем жалоба на поломку («но опять сломалось через день»), то итоговая тональность — **`negative`**.

In [6]:
mixed_df = dev[dev['tags'].fillna('').str.contains('mixed')]
print(f'Всего смешанных сообщений в dev: {len(mixed_df)}')
print('Распределение тональности в mixed:')
print(mixed_df['sentiment'].value_counts())

print('\nПримеры смешанных сообщений:')
for idx, r in mixed_df.head(4).iterrows():
    print(f'- [{r["sentiment"]}] {r["text"]}')

## 6. Разбор сложных тегов: `typos`, `emoji`, `context`
- `typos`: пропуск букв, слитное написание, отсутствие знаков препинания («добролись», «проходчт»).
- `emoji`: эмоциональный акцент выражен смайликами (`😡`, `🙏`, `👍`).
- `context`: смысл понятен только из контекста диалога (например, согласие или уточнение).

In [7]:
for tag in ['typos', 'emoji', 'context']:
    sub = dev[dev['tags'].fillna('').str.contains(tag)]
    print(f'=== Тег: {tag} (Всего: {len(sub)}) ===')
    print(sub['sentiment'].value_counts())
    sample = sub.iloc[0]['text']
    print(f'Пример: {sample}\n')

## 7. Архитектурные правила и функции-эвристики для препроцессора и моделей

На основе анализа `labeling_guide.md` фиксируем 4 обязательных правила пайплайна:

1. **Правило иерархии служебных сообщений:**
   ```python
   if is_service == 1:
       sentiment = 'neutral'
       aggression = 0
   ```
2. **Правило вложенности агрессии:**
   ```python
   if sentiment != 'negative':
       aggression = 0
   ```
3. **Лексические маркеры агрессии и обсценности:**
   Обфусцированный мат (`руко@опые`, `х..`, `б***`) детектируется регулярными выражениями и признаками мата.
4. **Учет сарказма:**
   Сочетание слов благодарности («спасибо», «молодцы») с негативными эмодзи (`😡`, `🤦`) или фразами типа «опять не работает» должно смещать предсказание в сторону `negative`.